# 🏆 Football Goal Prediction — V6

**Changelog dari V5 → V6 (berdasarkan roadmap perbaikan):**

| Prioritas | Item | Estimasi Impact |
|---|---|---|
| 🔴 Kritis | **Hapus data leak GT** dari rolling features (model jujur) | — |
| 🔴 Kritis | **Jangan buang pre-1990**, pakai full data + recency weight | Tinggi |
| 🟠 Tinggi | **Feature engineering terpisah** per gender M vs W | Tinggi |
| 🟠 Tinggi | **Zero-Inflated / Hurdle model** (ZIP) untuk ~30% zero | Sedang-Tinggi |
| 🟡 Sedang | **Optuna** Bayesian tuning (dipertahankan dari V5) | Sedang |
| 🟡 Sedang | **Fitur home/away terpisah** + H2H diperluas | Sedang |
| 🟡 Sedang | **Missing value flags** untuk rank & GDP | Sedang |
| 🟢 Minor | **Stacking** LGB + XGB + CatBoost + Ridge meta-learner | Sedang |
| 🟢 Minor | **ZIP-aware MBR** + per-gender Dixon-Coles ρ | Kecil |

**Target:** AW-MAE < 2.0

## 1. Setup & Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

from scipy.stats import poisson
from scipy.optimize import minimize_scalar
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import joblib

pd.set_option('display.max_columns', 60)
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except Exception:
    pass

SEED = 42
np.random.seed(SEED)

MAX_GOALS = 8
MODEL_DIR  = './models_v6'
os.makedirs(MODEL_DIR, exist_ok=True)

# V6: Recency decay — exponential, half-life 4 years
HALF_LIFE_YEARS = 4.0

print('Libraries loaded ✓')
print('V6: No GT leak | No 1990 cutoff | Per-gender FE | ZIP Hurdle | Home/Away features | GDP flags')

## 2. Load Data

**V6 fix:** Tidak ada date cutoff — seluruh data historis digunakan. Recency weighting menangani pertandingan lama via exponential decay.

In [ ]:
train = pd.read_csv('../dataset/train.csv')
test  = pd.read_csv('../dataset/test.csv')
gt    = pd.read_csv('../dataset/test_with_groundtruth.csv')  # HANYA untuk evaluasi akhir

train['date'] = pd.to_datetime(train['date'])
test['date']  = pd.to_datetime(test['date'])

print(f'Train : {train.shape}  ({train["date"].min().date()} → {train["date"].max().date()})')
print(f'Test  : {test.shape}   ({test["date"].min().date()} → {test["date"].max().date()})')
print(f'GT    : {gt.shape}')
print()
# V6: Data pre-1990 masih digunakan, hanya diberi bobot lebih rendah via recency decay
pre_1990 = (train['date'] < '1990-01-01').sum()
print(f'Data pre-1990: {pre_1990} baris  (V5 membuang ini, V6 mempertahankan dengan bobot rendah)')

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title in zip(axes, ['team_goals', 'opp_goals'], ['Team Goals', 'Opp Goals']):
    counts = train[col].value_counts().sort_index().head(12)
    ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
    ax.set_title(f'{title} Distribution (Full Train)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Goals'); ax.set_ylabel('Count')
    for i, v in zip(counts.index, counts.values):
        ax.text(i, v + 100, str(v), ha='center', fontsize=8)
plt.tight_layout(); plt.show()

zero_rate_team = (train['team_goals'] == 0).mean()
zero_rate_opp  = (train['opp_goals']  == 0).mean()
print(f'Zero rate — team_goals: {zero_rate_team:.3f} ({100*zero_rate_team:.1f}%)')
print(f'Zero rate — opp_goals : {zero_rate_opp:.3f}  ({100*zero_rate_opp:.1f}%)')
print(f'→ Motivasi ZIP/Hurdle model: ~{100*(zero_rate_team+zero_rate_opp)/2:.0f}% observasi = 0 gol')

In [ ]:
for g in ['M', 'W']:
    sub = train[train['gender'] == g]
    print(f'Gender {g}: n={len(sub):6d}  '
          f'mean_team_goals={sub["team_goals"].mean():.3f}  '
          f'mean_opp_goals={sub["opp_goals"].mean():.3f}  '
          f'P(draw)={(sub["team_goals"] == sub["opp_goals"]).mean():.3f}  '
          f'P(0-0)={((sub["team_goals"]==0)&(sub["opp_goals"]==0)).mean():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, g, color in zip(axes, ['M', 'W'], ['#2980b9', '#e91e8c']):
    sub = train[train['gender'] == g]
    ax.hist(sub['team_goals'], bins=range(0, 10), color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'Team Goals — Gender {g}  (n={len(sub)})', fontweight='bold')
    ax.set_xlabel('Goals'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()

## 4. Definisi Metrik Resmi AW-MAE (Unchanged)

In [ ]:
def get_tournament_weight(tournament: str) -> float:
    t = str(tournament).lower().strip()
    if "fifa world cup" in t or t == "world cup":   return 2.00
    if "afc championship" in t or "afc asian cup" in t or "asian cup" in t: return 1.80
    if "friendly" in t: return 0.96
    return 1.20

EXACT_PENALTY            = 0.30
OUTCOME_PENALTY          = 0.25
GD_PENALTY               = 0.15
WRONG_OUTCOME_MULTIPLIER = 1.50
NONLINEAR_POWER          = 1.50

def _outcome(a: int, b: int) -> int:
    if a > b:  return  1
    if a < b:  return -1
    return 0

def official_match_loss(y_team_true, y_opp_true, y_team_pred, y_opp_pred):
    y_team_true = np.asarray(y_team_true).astype(int)
    y_opp_true  = np.asarray(y_opp_true).astype(int)
    y_team_pred = np.asarray(y_team_pred).astype(int)
    y_opp_pred  = np.asarray(y_opp_pred).astype(int)
    base_mae = (np.abs(y_team_true - y_team_pred) + np.abs(y_opp_true - y_opp_pred)) / 2.0
    exact_hit    = (y_team_true == y_team_pred) & (y_opp_true == y_opp_pred)
    true_outcome = np.vectorize(_outcome)(y_team_true, y_opp_true)
    pred_outcome = np.vectorize(_outcome)(y_team_pred, y_opp_pred)
    outcome_hit  = (true_outcome == pred_outcome)
    gd_hit       = ((y_team_true - y_opp_true) == (y_team_pred - y_opp_pred))
    penalty = ((~exact_hit).astype(float)   * EXACT_PENALTY +
               (~outcome_hit).astype(float) * OUTCOME_PENALTY +
               (~gd_hit).astype(float)      * GD_PENALTY)
    raw_error = base_mae + penalty
    raw_error = np.where(outcome_hit, raw_error, raw_error * WRONG_OUTCOME_MULTIPLIER)
    return raw_error ** NONLINEAR_POWER

def awmae_score(y_team_true, y_opp_true, y_team_pred, y_opp_pred, weights=None) -> float:
    losses = official_match_loss(y_team_true, y_opp_true, y_team_pred, y_opp_pred)
    if weights is None: weights = np.ones(len(losses), dtype=float)
    else: weights = np.asarray(weights, dtype=float)
    return float(np.average(losses, weights=weights))

print('AW-MAE metric defined ✓ (unchanged)')

## 5. Feature Engineering V6

**Perbaikan utama:**
1. **Tidak ada data leak** — `reconstruct_core_features` tidak menggunakan GT test
2. **Home/Away terpisah** — rolling stats dipisah konteks home vs away
3. **Extended H2H** — tambah window 10 dan win rate
4. **GDP missing flags** — flag eksplisit untuk rank & GDP yang hilang
5. **Per-gender history packs** — statistik historis dihitung terpisah per gender

In [ ]:
# ─── Time + Tournament features ────────────────────────────────────────
def add_time_features(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['year']       = df['date'].dt.year.astype(int)
    df['month']      = df['date'].dt.month.astype(int)
    df['quarter']    = df['date'].dt.quarter.astype(int)
    df['dayofweek']  = df['date'].dt.dayofweek.astype(int)
    df['dayofyear']  = df['date'].dt.dayofyear.astype(int)
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['month_sin']  = np.sin(2 * np.pi * df['month']    / 12.0)
    df['month_cos']  = np.cos(2 * np.pi * df['month']    / 12.0)
    df['dow_sin']    = np.sin(2 * np.pi * df['dayofweek'] / 7.0)
    df['dow_cos']    = np.cos(2 * np.pi * df['dayofweek'] / 7.0)
    df['doy_sin']    = np.sin(2 * np.pi * df['dayofyear'] / 366.0)
    df['doy_cos']    = np.cos(2 * np.pi * df['dayofyear'] / 366.0)
    return df

def assign_tournament_importance(name):
    if not isinstance(name, str) or name.strip() == '': return 1
    n = name.lower()
    if 'fifa world cup' in n and 'qualif' not in n: return 5
    tier4 = ['uefa euro','copa america','copa américa','african cup of nations',
             'africa cup of nations','afc asian cup','gold cup','concacaf championship',
             'ofc nations cup','oceania nations cup','confederations cup']
    if any(k in n for k in tier4) and 'qualif' not in n: return 4
    if 'qualif' in n or 'nations league' in n or 'olympic' in n: return 3
    tier2 = ['cecafa','cosafa','cfu caribbean','gulf cup','aff championship',
             'southeast asian',"king's cup",'merdeka','island games','asian games',
             'south asian','baltic cup','nordic championship','british home championship',
             'algarve cup','cyprus cup','shebelieves','tournament','cup']
    if any(k in n for k in tier2): return 2
    return 1

print('Time + tournament importance functions defined ✓')

In [ ]:
# ─── Last-known rank lookup (dari full train, termasuk pre-1990) ──────────
def build_last_known_rank(raw_train_df):
    raw = raw_train_df.copy()
    raw['date'] = pd.to_datetime(raw['date'])
    team_side = (raw[['date','team','rank_team','confederation_team']]
                 .rename(columns={'team':'name','rank_team':'rank','confederation_team':'conf'})
                 .dropna(subset=['rank']))
    opp_side  = (raw[['date','opponent','rank_opponent','confederation_opp']]
                 .rename(columns={'opponent':'name','rank_opponent':'rank','confederation_opp':'conf'})
                 .dropna(subset=['rank']))
    stacked = pd.concat([team_side, opp_side], ignore_index=True).sort_values('date')
    last_rank = stacked.groupby('name').last()[['rank','conf']].rename(columns={'rank':'last_rank'})
    conf_median = last_rank.groupby('conf')['last_rank'].median()
    last_rank['conf_median_rank'] = last_rank['conf'].map(conf_median)
    last_rank['last_rank_filled'] = (last_rank['last_rank']
        .fillna(last_rank['conf_median_rank'])
        .fillna(last_rank['last_rank'].median()))
    return last_rank[['last_rank_filled']].to_dict()['last_rank_filled']

RANK_LOOKUP        = build_last_known_rank(train)
GLOBAL_MEDIAN_RANK = np.median(list(RANK_LOOKUP.values()))
print(f'Last-known rank lookup: {len(RANK_LOOKUP)} teams | global median: {GLOBAL_MEDIAN_RANK:.0f}')

In [ ]:
# ─── Rolling form features (shift-1 anti-leak) ────────────────────────
def _match_outcome_cols(df):
    df = df.copy()
    tg = df['team_goals'].astype(float)
    og = df['opp_goals'].astype(float)
    df['_points'] = np.where(tg > og, 3, np.where(tg == og, 1, 0))
    df['_gd']     = tg - og
    df['_win']    = (tg > og).astype(float)
    df['_draw']   = (tg == og).astype(float)
    df['_loss']   = (tg < og).astype(float)
    return df

def add_rolling_team_features(df):
    df = _match_outcome_cols(df)
    def lr(col, window, op):
        return df.groupby('team', sort=False)[col].transform(
            lambda s: s.shift(1).rolling(window, min_periods=1).agg(op))
    df['team_points_last5']       = lr('_points', 5, 'sum')
    df['team_points_last10']      = lr('_points', 10, 'sum')
    df['team_gd_last5']           = lr('_gd', 5, 'sum')
    df['team_avg_goals_last5']    = lr('team_goals', 5, 'mean')
    df['team_avg_conceded_last5'] = lr('opp_goals',  5, 'mean')
    df['team_win_rate_last10']    = lr('_win', 10, 'mean')
    df['days_since_last_match_team'] = df.groupby('team', sort=False)['date'].transform(
        lambda s: (s - s.shift(1)).dt.days)
    # Streak features
    for col_name, res_col in [('streak_win','_win'),('streak_loss','_loss'),('streak_draw','_draw')]:
        df[f'team_{col_name}'] = df.groupby('team', sort=False)[res_col].transform(
            lambda s: (s.shift(1).groupby((s.shift(1) != s.shift(1).shift(1)).cumsum()).cumcount() + 1)
                      * s.shift(1)
        ).fillna(0)
    return df

print('Rolling team features defined ✓')

In [ ]:
# ─── V6 NEW: Home/Away split rolling features ─────────────────────────
def add_home_away_rolling_features(df):
    """
    V6: Rolling stats dipisah berdasarkan konteks home vs away.
    Ini penting karena performa tim bisa sangat berbeda di kandang vs tandang.
    """
    df = df.copy()
    if '_win' not in df.columns:
        tg = df['team_goals'].astype(float)
        og = df['opp_goals'].astype(float)
        df['_win'] = (tg > og).astype(float)

    for ctx_name, ctx_val in [('home', 1), ('away', 0)]:
        # Mask hanya baris dalam konteks ini (NaN untuk baris lainnya)
        mask = (df['is_home'] == ctx_val)
        if ctx_name == 'away':
            # Away = bukan home DAN bukan neutral
            mask = (df['is_home'] == 0) & (df['neutral'] == 0)

        tmp_tg  = df['team_goals'].where(mask, np.nan)
        tmp_win = df['_win'].where(mask, np.nan)

        # Assign sementara untuk group transform
        df[f'_tmp_tg_{ctx_name}']  = tmp_tg
        df[f'_tmp_win_{ctx_name}'] = tmp_win

        df[f'team_{ctx_name}_avg_goals_last5'] = df.groupby('team', sort=False)[f'_tmp_tg_{ctx_name}'].transform(
            lambda s: s.shift(1).rolling(5, min_periods=1).mean()
        )
        df[f'team_{ctx_name}_win_rate_last5'] = df.groupby('team', sort=False)[f'_tmp_win_{ctx_name}'].transform(
            lambda s: s.shift(1).rolling(5, min_periods=1).mean()
        )

    # Bersihkan kolom temp
    df = df.drop(columns=[c for c in df.columns if c.startswith('_tmp_')], errors='ignore')

    # Home advantage: selisih performa home vs away
    df['team_home_vs_away_goals'] = (
        df['team_home_avg_goals_last5'].fillna(0) - df['team_away_avg_goals_last5'].fillna(0))
    df['team_home_vs_away_winrate'] = (
        df['team_home_win_rate_last5'].fillna(0) - df['team_away_win_rate_last5'].fillna(0))

    return df

print('Home/Away split rolling features defined ✓')

In [ ]:
# ─── Opponent-side mirror + extended H2H ──────────────────────────────
HOME_AWAY_PAIRS = [
    ('team_home_avg_goals_last5',   'opp_home_avg_goals_last5'),
    ('team_away_avg_goals_last5',   'opp_away_avg_goals_last5'),
    ('team_home_win_rate_last5',    'opp_home_win_rate_last5'),
    ('team_away_win_rate_last5',    'opp_away_win_rate_last5'),
    ('team_home_vs_away_goals',     'opp_home_vs_away_goals'),
    ('team_home_vs_away_winrate',   'opp_home_vs_away_winrate'),
]

ROLLING_PAIRS = [
    ('team_points_last5',        'opp_points_last5'),
    ('team_points_last10',       'opp_points_last10'),
    ('team_gd_last5',            'opp_gd_last5'),
    ('team_avg_goals_last5',     'opp_avg_goals_last5'),
    ('team_avg_conceded_last5',  'opp_avg_conceded_last5'),
    ('team_win_rate_last10',     'opp_win_rate_last10'),
    ('days_since_last_match_team', 'days_since_last_match_opp'),
    ('team_streak_win',          'opp_streak_win'),
    ('team_streak_loss',         'opp_streak_loss'),
    ('team_streak_draw',         'opp_streak_draw'),
]

def add_opponent_side_features(df):
    all_pairs = ROLLING_PAIRS + HOME_AWAY_PAIRS
    src_cols = [p[0] for p in all_pairs if p[0] in df.columns]
    all_pairs_avail = [(p[0], p[1]) for p in all_pairs if p[0] in df.columns]
    cols_needed = ['match_id','team'] + src_cols
    mirror = df[[c for c in cols_needed if c in df.columns]].rename(
        columns={'team':'opponent', **{p[0]: p[1] for p in all_pairs_avail}})
    return df.merge(mirror, on=['match_id','opponent'], how='left')

def add_h2h_features(df):
    """V6: Extended H2H — tambah window 10 dan win rate."""
    df = df.copy()
    if '_points' not in df.columns or '_gd' not in df.columns:
        df = _match_outcome_cols(df)
    g = df.groupby(['team','opponent'], sort=False)
    df['h2h_points_last5']   = g['_points'].transform(lambda s: s.shift(1).rolling(5,  min_periods=1).sum())
    df['h2h_points_last10']  = g['_points'].transform(lambda s: s.shift(1).rolling(10, min_periods=1).sum())
    df['h2h_gd_last5']       = g['_gd'].transform(    lambda s: s.shift(1).rolling(5,  min_periods=1).sum())
    df['h2h_gd_last10']      = g['_gd'].transform(    lambda s: s.shift(1).rolling(10, min_periods=1).sum())
    df['h2h_win_rate_last10'] = g['_win'].transform(  lambda s: s.shift(1).rolling(10, min_periods=1).mean())
    df['h2h_goals_scored_last5'] = g['team_goals'].transform(
        lambda s: s.shift(1).rolling(5, min_periods=1).mean())
    df['h2h_goals_conceded_last5'] = g['opp_goals'].transform(
        lambda s: s.shift(1).rolling(5, min_periods=1).mean())
    return df

print('Opponent side + Extended H2H defined ✓')

In [ ]:
# ─── Elo ratings (unchanged) ─────────────────────────────────────────
def _elo_match_weight_K(tournament, match_date):
    if not isinstance(tournament, str): return 30
    n = tournament.lower()
    if 'olympic' in n:
        year = pd.Timestamp(match_date).year
        return 60 if 1908 <= year <= 1980 else 30
    if 'fifa world cup' in n and 'qualif' not in n: return 60
    tier50 = ['uefa euro','copa america','copa américa','african cup of nations',
              'africa cup of nations','afc asian cup','gold cup','concacaf championship',
              'ofc nations cup','oceania nations cup','confederations cup']
    if any(k in n for k in tier50) and 'qualif' not in n: return 50
    if 'qualif' in n: return 40
    if 'nations league' in n: return 40
    if 'friendly' in n: return 20
    return 30

def _elo_goal_diff_multiplier(gd):
    gd = int(abs(gd))
    if gd <= 1: return 1.0
    if gd == 2: return 1.5
    return (11.0 + gd) / 8.0

def add_elo_features(df, initial=1500.0, home_adv=100.0):
    df = df.copy()
    elo_per_gender = {'M': {}, 'W': {}}
    pre_elo = {}
    first_rows = (df.drop_duplicates('match_id', keep='first')
                    .sort_values(['date','match_id']).reset_index(drop=True))
    for row in first_rows.itertuples(index=False):
        elo = elo_per_gender.get(row.gender)
        if elo is None: continue
        R_team = elo.get(row.team,     initial)
        R_opp  = elo.get(row.opponent, initial)
        pre_elo[(row.match_id, row.team)]     = R_team
        pre_elo[(row.match_id, row.opponent)] = R_opp
        tg, og = row.team_goals, row.opp_goals
        if pd.isna(tg) or pd.isna(og): continue
        tg, og = int(tg), int(og)
        if row.neutral:        HA = 0.0
        elif row.is_home == 1: HA = +home_adv
        else:                  HA = -home_adv
        dr = R_team - R_opp + HA
        We = 1.0 / (10 ** (-dr / 400.0) + 1.0)
        W  = 1.0 if tg > og else (0.0 if tg < og else 0.5)
        K  = _elo_match_weight_K(row.tournament, row.date)
        G  = _elo_goal_diff_multiplier(tg - og)
        P  = round(K * G * (W - We))
        elo[row.team]     = R_team + P
        elo[row.opponent] = R_opp  - P
    pre_elo_df = pd.DataFrame(
        [(mid, name, val) for (mid, name), val in pre_elo.items()],
        columns=['match_id','_name','elo'])
    df = df.merge(pre_elo_df.rename(columns={'_name':'team','elo':'elo_team'}),
                  on=['match_id','team'], how='left')
    df = df.merge(pre_elo_df.rename(columns={'_name':'opponent','elo':'elo_opponent'}),
                  on=['match_id','opponent'], how='left')
    df['elo_team']     = df['elo_team'].fillna(initial)
    df['elo_opponent'] = df['elo_opponent'].fillna(initial)
    return df

print('Elo feature function defined ✓')

In [ ]:
# ─── Pi-ratings ──────────────────────────────────────────────────────
def _pi_error_smooth(e):
    return 3.0 * np.log10(1.0 + abs(e)) * np.sign(e)

def add_pi_rating_features(df, lambda_w=0.06, gamma_w=0.5, initial=0.0):
    df = df.copy()
    pi_home = {'M': {}, 'W': {}}
    pi_away = {'M': {}, 'W': {}}
    pre = {}
    first_rows = (df.drop_duplicates('match_id', keep='first')
                    .sort_values(['date','match_id']).reset_index(drop=True))
    for row in first_rows.itertuples(index=False):
        g = row.gender
        if g not in pi_home: continue
        team, opp = row.team, row.opponent
        H_t = pi_home[g].get(team, initial); A_t = pi_away[g].get(team, initial)
        H_o = pi_home[g].get(opp,  initial); A_o = pi_away[g].get(opp,  initial)
        pre[(row.match_id, team)] = (H_t, A_t)
        pre[(row.match_id, opp)]  = (H_o, A_o)
        tg, og = row.team_goals, row.opp_goals
        if pd.isna(tg) or pd.isna(og): continue
        tg, og = int(tg), int(og)
        if row.neutral:
            expected_gd = 0.5 * ((H_t - A_o) + (A_t - H_o))
        elif row.is_home == 1:
            expected_gd = H_t - A_o
        else:
            expected_gd = A_t - H_o
        error = (tg - og) - expected_gd
        delta = lambda_w * _pi_error_smooth(error)
        if row.neutral:
            pi_home[g][team] = H_t + 0.5*delta; pi_away[g][team] = A_t + 0.5*delta
            pi_home[g][opp]  = H_o - 0.5*delta; pi_away[g][opp]  = A_o - 0.5*delta
        elif row.is_home == 1:
            new_H_t = H_t + delta
            pi_home[g][team] = new_H_t; pi_away[g][team] = A_t + gamma_w*(new_H_t - H_t)
            new_A_o = A_o - delta
            pi_away[g][opp]  = new_A_o; pi_home[g][opp]  = H_o + gamma_w*(new_A_o - A_o)
        else:
            new_A_t = A_t + delta
            pi_away[g][team] = new_A_t; pi_home[g][team] = H_t + gamma_w*(new_A_t - A_t)
            new_H_o = H_o - delta
            pi_home[g][opp]  = new_H_o; pi_away[g][opp]  = A_o + gamma_w*(new_H_o - H_o)
    pre_df = pd.DataFrame(
        [(mid, name, h, a) for (mid, name), (h, a) in pre.items()],
        columns=['match_id','_name','pi_home','pi_away'])
    df = df.merge(pre_df.rename(columns={'_name':'team','pi_home':'pi_home_team','pi_away':'pi_away_team'}),
                  on=['match_id','team'], how='left')
    df = df.merge(pre_df.rename(columns={'_name':'opponent','pi_home':'pi_home_opp','pi_away':'pi_away_opp'}),
                  on=['match_id','opponent'], how='left')
    for c in ['pi_home_team','pi_away_team','pi_home_opp','pi_away_opp']:
        df[c] = df[c].fillna(initial)
    return df

print('Pi-rating function defined ✓')

In [ ]:
# ─── V6: Reconstruct core features TANPA data leak ────────────────────
RECONSTRUCTED_COLS = [
    'team_points_last5','team_points_last10',
    'opp_points_last5','opp_points_last10',
    'team_gd_last5','opp_gd_last5',
    'team_avg_goals_last5','team_avg_conceded_last5',
    'opp_avg_goals_last5','opp_avg_conceded_last5',
    'team_win_rate_last10','opp_win_rate_last10',
    'points_last5_diff','gd_last5_diff',
    'h2h_points_last5','h2h_points_last10','h2h_gd_last5','h2h_gd_last10',
    'h2h_win_rate_last10','h2h_goals_scored_last5','h2h_goals_conceded_last5',
    'days_since_last_match_team','days_since_last_match_opp',
    'elo_team','elo_opponent',
    'pi_home_team','pi_away_team','pi_home_opp','pi_away_opp',
    'team_streak_win','team_streak_loss','team_streak_draw',
    'opp_streak_win','opp_streak_loss','opp_streak_draw',
    # V6 NEW: Home/Away split
    'team_home_avg_goals_last5','team_away_avg_goals_last5',
    'team_home_win_rate_last5','team_away_win_rate_last5',
    'team_home_vs_away_goals','team_home_vs_away_winrate',
    'opp_home_avg_goals_last5','opp_away_avg_goals_last5',
    'opp_home_win_rate_last5','opp_away_win_rate_last5',
    'opp_home_vs_away_goals','opp_home_vs_away_winrate',
]

def reconstruct_core_features(train_df, test_df):
    """
    V6 FIX: Tidak menggunakan ground truth test dalam rolling features.
    Test rows memiliki team_goals/opp_goals = NaN → tidak bisa 'bocor' ke rolling window.
    """
    common = ['Id','match_id','date','gender','team','opponent',
              'is_home','neutral','tournament','team_goals','opp_goals']

    tr_core = train_df[common].copy(); tr_core['_split'] = 'train'

    # V6 CRITICAL: Test goals = NaN (tidak inject ground truth)
    te_core = test_df[[c for c in common if c in test_df.columns]].copy()
    if 'team_goals' not in te_core.columns: te_core['team_goals'] = np.nan
    if 'opp_goals'  not in te_core.columns: te_core['opp_goals']  = np.nan
    te_core['_split'] = 'test'

    hist = pd.concat([tr_core, te_core], ignore_index=True)
    hist['date'] = pd.to_datetime(hist['date'])
    hist = hist.sort_values(['date','match_id','is_home'],
                            ascending=[True,True,False]).reset_index(drop=True)

    # Rolling features — NaN test goals tidak bocor ke rolling window
    hist = add_rolling_team_features(hist)
    hist = add_home_away_rolling_features(hist)   # V6 NEW
    hist = add_opponent_side_features(hist)
    hist['points_last5_diff'] = hist['team_points_last5'] - hist['opp_points_last5']
    hist['gd_last5_diff']     = hist['team_gd_last5']     - hist['opp_gd_last5']
    hist = add_h2h_features(hist)
    hist = add_elo_features(hist)
    hist = add_pi_rating_features(hist)
    hist['tournament_importance'] = hist['tournament'].map(assign_tournament_importance)

    drop_tmp = [c for c in ['_points','_gd','_win','_draw','_loss'] if c in hist.columns]
    hist = hist.drop(columns=drop_tmp, errors='ignore')

    new_cols = [c for c in RECONSTRUCTED_COLS if c in hist.columns] + ['tournament_importance']
    tr_out = train_df.drop(columns=[c for c in new_cols if c in train_df.columns], errors='ignore').merge(
        hist.loc[hist['_split']=='train', ['Id']+new_cols], on='Id', how='left')
    te_out = test_df.merge(
        hist.loc[hist['_split']=='test',  ['Id']+new_cols], on='Id', how='left')
    return tr_out, te_out

print('reconstruct_core_features (NO DATA LEAK) defined ✓')

In [ ]:
# ─── V6: Per-gender history pack ──────────────────────────────────────
def build_history_pack(train_df, gender=None):
    """
    V6: Membangun statistik historis terpisah per gender.
    M dan W punya distribusi gol berbeda — mixing bias features.
    """
    tr = add_time_features(train_df)
    if gender is not None:
        tr = tr[tr['gender'] == gender].copy()
    tr = tr.sort_values(['date','match_id','team']).copy()

    team_hist = (tr.groupby('team')
                   .agg(team_hist_avg_goals  =('team_goals','mean'),
                        team_hist_avg_conceded=('opp_goals','mean'),
                        team_hist_std_goals   =('team_goals','std'),
                        team_hist_matches     =('match_id','nunique'))
                   .reset_index())
    tournament_hist = (tr.groupby('tournament')
                         .agg(tournament_avg_goals=('team_goals','mean'),
                              tournament_avg_total =('team_goals',
                                  lambda s: (s + tr.loc[s.index,'opp_goals']).mean()))
                         .reset_index())
    venue_hist = (tr.groupby('venue_country')
                    .agg(venue_avg_goals=('team_goals','mean'),
                         venue_avg_total=('team_goals',
                             lambda s: (s + tr.loc[s.index,'opp_goals']).mean()))
                    .reset_index())
    conf_hist = (tr.groupby('confederation_team')
                   .agg(conf_team_avg_goals    =('team_goals','mean'),
                        conf_team_avg_conceded  =('opp_goals','mean'))
                   .reset_index())
    gender_hist = (tr.groupby('gender')
                     .agg(gender_avg_goals=('team_goals','mean'),
                          gender_avg_total=('team_goals',
                              lambda s: (s + tr.loc[s.index,'opp_goals']).mean()))
                     .reset_index())
    return {'team_hist':tournament_hist,'tournament_hist':tournament_hist,
            'venue_hist':venue_hist,'conf_hist':conf_hist,'gender_hist':gender_hist,
            'team_hist_direct':team_hist}

print('Per-gender history pack function defined ✓')

In [ ]:
# ─── V6: engineer_features dengan GDP missing flags ───────────────────
def engineer_features(df, history_pack):
    """
    V6: Ditambahkan:
    - rank_missing flag (sudah ada di V5)
    - gdp_missing flag (V6 NEW)
    - population_missing flag (V6 NEW)
    - pi × tournament interaction
    - altitude interactions
    """
    d = add_time_features(df).copy()

    if 'altitude_venue' in d.columns:
        d['altitude_venue'] = d['altitude_venue'].replace(-9999, np.nan)

    d['tournament_weight'] = d['tournament'].apply(get_tournament_weight)

    # History stats dari per-gender pack
    team_hist = history_pack['team_hist_direct']
    opp_team_hist = team_hist.rename(columns={
        'team':'opponent',
        'team_hist_avg_goals':    'opp_hist_avg_goals',
        'team_hist_avg_conceded': 'opp_hist_avg_conceded',
        'team_hist_std_goals':    'opp_hist_std_goals',
        'team_hist_matches':      'opp_hist_matches',
    })
    d = d.merge(team_hist,           on='team',            how='left')
    d = d.merge(opp_team_hist,       on='opponent',        how='left')
    d = d.merge(history_pack['tournament_hist'], on='tournament',    how='left')
    d = d.merge(history_pack['venue_hist'],      on='venue_country', how='left')
    d = d.merge(history_pack['conf_hist'],       on='confederation_team', how='left')
    d = d.merge(history_pack['gender_hist'],     on='gender',        how='left')

    # Rank proxy
    d['rank_proxy_team'] = d['team'].map(RANK_LOOKUP).fillna(GLOBAL_MEDIAN_RANK)
    d['rank_proxy_opp']  = d['opponent'].map(RANK_LOOKUP).fillna(GLOBAL_MEDIAN_RANK)
    d['rank_proxy_diff'] = d['rank_proxy_team'] - d['rank_proxy_opp']
    d['rank_proxy_gap']  = np.abs(d['rank_proxy_diff'])
    # V5 flag rank (keep)
    d['rank_proxy_missing_team'] = (~d['team'].isin(RANK_LOOKUP)).astype(int)
    d['rank_proxy_missing_opp']  = (~d['opponent'].isin(RANK_LOOKUP)).astype(int)

    # V6 NEW: GDP missing flag
    if 'gdp_per_capita_team' in d.columns:
        d['gdp_missing_team'] = d['gdp_per_capita_team'].isna().astype(int)
    if 'gdp_per_capita_opp' in d.columns:
        d['gdp_missing_opp']  = d['gdp_per_capita_opp'].isna().astype(int)
    # V6 NEW: Population missing flag
    if 'population_team' in d.columns:
        d['population_missing_team'] = d['population_team'].isna().astype(int)
    if 'population_opp' in d.columns:
        d['population_missing_opp']  = d['population_opp'].isna().astype(int)

    # Elo derived
    d['elo_diff']    = d['elo_team'] - d['elo_opponent']
    d['elo_sum']     = d['elo_team'] + d['elo_opponent']
    d['elo_gap_abs'] = np.abs(d['elo_diff'])

    # Pi-rating derived
    d['pi_effective_team'] = np.where(d['neutral']==1,
        0.5*(d['pi_home_team'] + d['pi_away_team']),
        np.where(d['is_home']==1, d['pi_home_team'], d['pi_away_team']))
    d['pi_effective_opp']  = np.where(d['neutral']==1,
        0.5*(d['pi_home_opp']  + d['pi_away_opp']),
        np.where(d['is_home']==1, d['pi_away_opp'], d['pi_home_opp']))
    d['pi_diff']    = d['pi_effective_team'] - d['pi_effective_opp']
    d['pi_gap_abs'] = np.abs(d['pi_diff'])

    # Form interactions
    d['rest_diff']          = d['days_since_last_match_team'] - d['days_since_last_match_opp']
    d['goal_form_diff']     = d['team_avg_goals_last5']   - d['opp_avg_goals_last5']
    d['conceded_form_diff'] = d['team_avg_conceded_last5'] - d['opp_avg_conceded_last5']
    d['attack_vs_defense']  = d['team_avg_goals_last5']   - d['opp_avg_conceded_last5']
    d['defense_vs_attack']  = d['opp_avg_goals_last5']    - d['team_avg_conceded_last5']
    d['winrate_diff']       = d['team_win_rate_last10'] - d['opp_win_rate_last10']
    d['h2h_strength']       = d['h2h_points_last5']    + 0.25 * d['h2h_gd_last5']
    d['same_confederation'] = (d['confederation_team'] == d['confederation_opp']).astype(int)
    d['major_tournament']   = (d['tournament_weight'] >= 1.8).astype(int)
    d['home_x_elo']         = d['is_home'] * d['elo_diff']
    d['neutral_x_elo']      = d['neutral'] * d['elo_diff']
    d['home_x_pi']          = d['is_home'] * d['pi_diff']
    d['neutral_x_pi']       = d['neutral'] * d['pi_diff']

    if 'distance_travel_team' in d.columns and 'distance_travel_opp' in d.columns:
        d['travel_diff'] = d['distance_travel_team'] - d['distance_travel_opp']
    if 'gdp_per_capita_team' in d.columns and 'gdp_per_capita_opp' in d.columns:
        d['gdp_diff'] = d['gdp_per_capita_team'] - d['gdp_per_capita_opp']
    if 'temperature_venue' in d.columns:
        d['temperature_home'] = d['temperature_venue'] * d['is_home']

    # Altitude interactions
    d['altitude_home_adv']    = d['altitude_venue'].fillna(0) * d['is_home']
    d['altitude_away_disadv'] = d['altitude_venue'].fillna(0) * (1-d['is_home'])*(1-d['neutral'])

    # Pi × tournament importance
    d['pi_diff_x_tournament']  = d['pi_diff']  * d['tournament_importance']
    d['elo_diff_x_tournament'] = d['elo_diff'] * d['tournament_importance']
    d['rank_x_elo_diff']       = d['rank_proxy_diff'] * d['elo_diff']

    # Streak diff
    d['streak_win_diff']  = d.get('team_streak_win',  pd.Series(0, index=d.index)).fillna(0) - \
                            d.get('opp_streak_win',   pd.Series(0, index=d.index)).fillna(0)
    d['streak_loss_diff'] = d.get('team_streak_loss', pd.Series(0, index=d.index)).fillna(0) - \
                            d.get('opp_streak_loss',  pd.Series(0, index=d.index)).fillna(0)

    # Form momentum
    d['form_momentum']     = d['team_points_last5'] - (d['team_points_last10'] - d['team_points_last5'])
    d['opp_form_momentum'] = d['opp_points_last5']  - (d['opp_points_last10']  - d['opp_points_last5'])
    d['momentum_diff']     = d['form_momentum'] - d['opp_form_momentum']

    # V6: Home/Away differential features
    if 'team_home_avg_goals_last5' in d.columns and 'opp_away_avg_goals_last5' in d.columns:
        d['home_attack_vs_away_defense'] = \
            d['team_home_avg_goals_last5'].fillna(0) - d['opp_away_avg_goals_last5'].fillna(0)
    if 'team_away_avg_goals_last5' in d.columns and 'opp_home_avg_goals_last5' in d.columns:
        d['away_attack_vs_home_defense'] = \
            d['team_away_avg_goals_last5'].fillna(0) - d['opp_home_avg_goals_last5'].fillna(0)

    # Log transforms
    for col in ['population_team','population_opp','gdp_per_capita_team','gdp_per_capita_opp',
                'distance_travel_team','distance_travel_opp']:
        if col in d.columns:
            d[f'log1p_{col}'] = np.log1p(np.clip(d[col], a_min=0, a_max=None))

    return d

print('engineer_features V6 defined ✓')

## 5f. Apply Feature Engineering V6

In [ ]:
print('Reconstructing core features (no data leak)...')
train_core, test_core = reconstruct_core_features(train, test)
print(f'Core rebuilt:  train={train_core.shape}  test={test_core.shape}')

# V6: Per-gender history packs
print('Building per-gender history packs...')
history_packs = {}
for g in ['M', 'W']:
    history_packs[g] = build_history_pack(train_core, gender=g)
print('Per-gender history packs built ✓')

# Apply features per gender, then concat
train_parts = []; test_parts = []
for g in ['M', 'W']:
    tr_g = train_core[train_core['gender'] == g].copy()
    te_g = test_core[test_core['gender'] == g].copy()
    train_parts.append(engineer_features(tr_g, history_packs[g]))
    test_parts.append(engineer_features(te_g,  history_packs[g]))

train_fe = pd.concat(train_parts, ignore_index=True)
test_fe  = pd.concat(test_parts,  ignore_index=True)

# Sort ulang sesuai urutan asli agar index align
train_fe = train_fe.sort_values('Id').reset_index(drop=True)
test_fe  = test_fe.sort_values('Id').reset_index(drop=True)

EXCLUDE = ['Id','match_id','date','team','opponent','venue_country','tournament',
           'gender','confederation_team','confederation_opp','team_goals','opp_goals']
common_cols = [c for c in train_fe.columns if c in test_fe.columns and c not in EXCLUDE]
feature_cols = sorted([c for c in common_cols
                       if pd.api.types.is_numeric_dtype(train_fe[c])
                       and pd.api.types.is_numeric_dtype(test_fe[c])])

train_fe[feature_cols] = train_fe[feature_cols].replace([np.inf,-np.inf], np.nan)
test_fe[feature_cols]  = test_fe[feature_cols].replace([np.inf,-np.inf], np.nan)
fill_map = train_fe[feature_cols].median(numeric_only=True)
train_fe[feature_cols] = train_fe[feature_cols].fillna(fill_map)
test_fe[feature_cols]  = test_fe[feature_cols].fillna(fill_map)

# V6: Exponential recency weight (no cutoff — seluruh data dipakai)
date_max  = train_fe['date'].max()
age_years = (date_max - train_fe['date']).dt.days / 365.25
train_fe['recency_weight'] = np.exp(-np.log(2) / HALF_LIFE_YEARS * age_years)
train_fe['fit_weight']     = train_fe['tournament_weight'] * train_fe['recency_weight']

print(f'\nTotal features V6: {len(feature_cols)}')
ha_feats = [c for c in feature_cols if 'home_avg' in c or 'away_avg' in c or 'home_win' in c or 'away_win' in c]
print(f'Home/Away split features ({len(ha_feats)}): {ha_feats}')
print(f'Recency weight range: [{train_fe["recency_weight"].min():.4f}, {train_fe["recency_weight"].max():.3f}]')
print(f'Pre-1990 data: {(train_fe["date"] < "1990-01-01").sum()} baris  (V5 membuang ini)')

## 6. Optuna Hyperparameter Tuning (LightGBM)

25 trials per gender per target pada fold-0 saja (cepat tapi terarah).

In [ ]:
def build_temporal_match_folds(df, n_splits=5, holdout_start_frac=0.55):
    meta = (df[['match_id','date']].drop_duplicates()
              .sort_values('date').reset_index(drop=True))
    start_idx = int(len(meta) * holdout_start_frac)
    eval_meta = meta.iloc[start_idx:].copy()
    blocks = np.array_split(eval_meta.index.to_numpy(), n_splits)
    folds = []
    for block in blocks:
        val_meta  = meta.loc[block].copy()
        val_start = val_meta['date'].min()
        val_ids   = set(val_meta['match_id'])
        tr_idx    = df.index[df['date'] < val_start].to_numpy()
        val_idx   = df.index[df['match_id'].isin(val_ids)].to_numpy()
        folds.append((tr_idx, val_idx))
    return folds

def enforce_match_consistency_vec(df_rows, team_pred, opp_pred):
    team_pred = np.asarray(team_pred, dtype=float)
    opp_pred  = np.asarray(opp_pred,  dtype=float)
    work = pd.DataFrame({'match_id':df_rows['match_id'].values,
                         'orig_idx':np.arange(len(team_pred)),
                         'team_pred':team_pred,'opp_pred':opp_pred})
    work['pair_order'] = work.groupby('match_id').cumcount()
    pair_counts = work.groupby('match_id')['pair_order'].transform('size')
    paired = work[pair_counts == 2]
    if len(paired) == 0: return team_pred, opp_pred
    wide = paired.pivot(index='match_id', columns='pair_order',
                        values=['team_pred','opp_pred','orig_idx'])
    tp0=wide[('team_pred',0)].values; tp1=wide[('team_pred',1)].values
    op0=wide[('opp_pred', 0)].values; op1=wide[('opp_pred', 1)].values
    idx0=wide[('orig_idx',0)].values.astype(int)
    idx1=wide[('orig_idx',1)].values.astype(int)
    a=0.5*(tp0+op1); b=0.5*(op0+tp1)
    team_adj=team_pred.copy(); opp_adj=opp_pred.copy()
    team_adj[idx0]=a; opp_adj[idx0]=b
    team_adj[idx1]=b; opp_adj[idx1]=a
    return team_adj, opp_adj

def round_clip(x, low=0, high=MAX_GOALS):
    return np.clip(np.rint(x), low, high).astype(int)

print('CV / consistency helpers defined ✓')

In [ ]:
X      = train_fe[feature_cols].astype(np.float32).values
y_team = train_fe['team_goals'].astype(np.float32).values
y_opp  = train_fe['opp_goals'].astype(np.float32).values
X_test = test_fe[feature_cols].astype(np.float32).values

official_weights_train = train_fe['tournament_weight'].values.astype(float)
fit_weights_train      = train_fe['fit_weight'].values.astype(float)
gender_train = train_fe['gender'].values
gender_test  = test_fe['gender'].values

def optuna_lgb_objective(trial, X_tr, y_tr, w_tr, X_val, y_val, w_val_aw):
    params = dict(
        objective='poisson', n_estimators=3000,
        learning_rate    =trial.suggest_float('learning_rate',     0.008, 0.05,  log=True),
        num_leaves       =trial.suggest_int(  'num_leaves',        31,    127),
        max_depth        =trial.suggest_int(  'max_depth',         4,     8),
        min_child_samples=trial.suggest_int(  'min_child_samples', 15,    60),
        subsample        =trial.suggest_float('subsample',         0.70,  0.95),
        subsample_freq=1,
        colsample_bytree =trial.suggest_float('colsample_bytree',  0.65,  0.95),
        reg_alpha        =trial.suggest_float('reg_alpha',         0.05,  2.0,  log=True),
        reg_lambda       =trial.suggest_float('reg_lambda',        0.5,   5.0,  log=True),
        random_state=SEED, verbose=-1)
    m = lgb.LGBMRegressor(**params)
    m.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(100, verbose=False)])
    pred = np.clip(m.predict(X_val), 1e-4, MAX_GOALS)
    return float(np.average(np.abs(y_val - pred), weights=w_val_aw))

def tune_lgb_per_gender(gender, n_trials=25):
    g_mask = gender_train == gender
    g_fe   = train_fe[g_mask].reset_index(drop=True)
    X_g    = X[g_mask]; yt_g=y_team[g_mask]; yo_g=y_opp[g_mask]
    w_g    = fit_weights_train[g_mask]; aw_g=official_weights_train[g_mask]
    folds_g = build_temporal_match_folds(g_fe, n_splits=5, holdout_start_frac=0.55)
    tr_idx, val_idx = folds_g[0]
    X_tr,X_val = X_g[tr_idx],X_g[val_idx]
    yt_tr=yt_g[tr_idx]; yo_tr=yo_g[tr_idx]
    w_tr=w_g[tr_idx];   aw_val=aw_g[val_idx]
    print(f'[{gender}] Tuning LGB — {n_trials} trials (train={len(tr_idx)}, val={len(val_idx)})')
    best_params = {}
    for target_name, y_tr_use, y_val_use in [('team',yt_tr,yt_g[val_idx]),
                                              ('opp', yo_tr,yo_g[val_idx])]:
        study = optuna.create_study(direction='minimize',
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(
            lambda trial: optuna_lgb_objective(trial,X_tr,y_tr_use,w_tr,X_val,y_val_use,aw_val),
            n_trials=n_trials, show_progress_bar=False)
        best_params[target_name] = study.best_params
        print(f'  [{gender}] {target_name}_goals val-MAE={study.best_value:.4f}  '
              f'num_leaves={study.best_params["num_leaves"]}  '
              f'lr={study.best_params["learning_rate"]:.4f}')
    return best_params

print('Optuna objective defined. Starting tuning...')
TUNED_LGB_PARAMS = {}
for g in ['M','W']:
    TUNED_LGB_PARAMS[g] = tune_lgb_per_gender(g, n_trials=25)
print('\nOptuna tuning complete ✓')

## 7. Zero-Inflated / Hurdle Model (V6 NEW)

**Motivasi:** ~30% observasi gol = 0. Model Poisson murni under-predicts zero probability.

**Strategi:**
- Untuk setiap gender, latih binary classifier LGB: P(goals = 0)
- Di MBR decoding (Section 9), gunakan ZIP PMF:  
  `P(Y=k) = p_zero*(k==0) + (1-p_zero)*Poisson(k; λ)`
- Ini meningkatkan prediksi 0-0, 1-0, 0-1 yang sangat berdampak pada AW-MAE

In [ ]:
# ─── V6 NEW: Zero-Inflated Hurdle Model ──────────────────────────────
n_train = len(train_fe); n_test = len(test_fe)

oof_p0_team  = np.zeros(n_train)
oof_p0_opp   = np.zeros(n_train)
test_p0_team = np.zeros(n_test)
test_p0_opp  = np.zeros(n_test)
oof_mask_hurdle = np.zeros(n_train, dtype=bool)

for gender in ['M','W']:
    g_tr_mask = gender_train == gender
    g_te_mask = gender_test  == gender
    if g_tr_mask.sum() < 200:
        print(f'[{gender}] insufficient rows, skip'); continue

    g_train_fe  = train_fe[g_tr_mask].reset_index(drop=True)
    X_g         = X[g_tr_mask]
    yt_g        = y_team[g_tr_mask]
    yo_g        = y_opp[g_tr_mask]
    X_test_g    = X_test[g_te_mask]
    orig_tr_idx = np.where(g_tr_mask)[0]
    orig_te_idx = np.where(g_te_mask)[0]
    folds_g     = build_temporal_match_folds(g_train_fe, n_splits=5, holdout_start_frac=0.55)

    test_p0_folds_team = []; test_p0_folds_opp = []

    for fold_i, (tr_idx, val_idx) in enumerate(folds_g, start=1):
        X_tr, X_val = X_g[tr_idx], X_g[val_idx]
        for y_arr, folds_list, oof_arr in [
            (yt_g, test_p0_folds_team, oof_p0_team),
            (yo_g, test_p0_folds_opp,  oof_p0_opp),
        ]:
            y_zero_tr  = (y_arr[tr_idx]  == 0).astype(int)
            y_zero_val = (y_arr[val_idx] == 0).astype(int)

            clf = lgb.LGBMClassifier(
                objective='binary', n_estimators=1000,
                learning_rate=0.05, num_leaves=31,
                subsample=0.80, colsample_bytree=0.80,
                min_child_samples=20,
                class_weight='balanced',   # tangani imbalance
                random_state=SEED, verbose=-1)
            clf.fit(X_tr, y_zero_tr,
                    eval_set=[(X_val, y_zero_val)],
                    callbacks=[lgb.early_stopping(50, verbose=False)])

            oof_arr[orig_tr_idx[val_idx]] = clf.predict_proba(X_val)[:, 1]
            folds_list.append(clf.predict_proba(X_test_g)[:, 1])

        oof_mask_hurdle[orig_tr_idx[val_idx]] = True

    test_p0_team[orig_te_idx] = np.mean(test_p0_folds_team, axis=0)
    test_p0_opp[orig_te_idx]  = np.mean(test_p0_folds_opp,  axis=0)

    g_oof_mask = oof_mask_hurdle & g_tr_mask
    actual_z_t  = (y_team[g_oof_mask] == 0).mean()
    actual_z_o  = (y_opp[g_oof_mask]  == 0).mean()
    pred_z_t    = oof_p0_team[g_oof_mask].mean()
    pred_z_o    = oof_p0_opp[g_oof_mask].mean()
    print(f'[{gender}] Actual P0: team={actual_z_t:.3f}, opp={actual_z_o:.3f}')
    print(f'[{gender}] Pred  P0: team={pred_z_t:.3f},   opp={pred_z_o:.3f}')

print('\nZIP Hurdle models trained ✓')

## 8. Full Model Training — Per-Gender × LGB + XGB + CatBoost

In [ ]:
def make_lgb_params(seed, gender='M', target='team'):
    p = TUNED_LGB_PARAMS.get(gender,{}).get(target,{})
    return dict(objective='poisson', n_estimators=4000,
        learning_rate    =p.get('learning_rate',    0.02),
        num_leaves       =p.get('num_leaves',        63),
        max_depth        =p.get('max_depth',         -1),
        min_child_samples=p.get('min_child_samples', 25),
        subsample        =p.get('subsample',         0.85), subsample_freq=1,
        colsample_bytree =p.get('colsample_bytree',  0.85),
        reg_alpha        =p.get('reg_alpha',          0.5),
        reg_lambda       =p.get('reg_lambda',         1.5),
        random_state=seed, verbose=-1)

def make_xgb_params(seed, gender='M', target='team'):
    p = TUNED_LGB_PARAMS.get(gender,{}).get(target,{})
    return dict(objective='count:poisson', n_estimators=3000,
        learning_rate   =p.get('learning_rate',    0.02),
        max_depth       =min(p.get('max_depth',     6),8),
        min_child_weight=max(p.get('min_child_samples',25)//10,2),
        subsample       =p.get('subsample',         0.85),
        colsample_bytree=p.get('colsample_bytree',  0.85),
        reg_alpha       =p.get('reg_alpha',          0.25),
        reg_lambda      =p.get('reg_lambda',         1.5),
        gamma=0.05, random_state=seed, tree_method='hist',
        eval_metric='mae', early_stopping_rounds=150, verbosity=0)

def make_cat_params(seed, gender='M', target='team'):
    p = TUNED_LGB_PARAMS.get(gender,{}).get(target,{})
    return dict(loss_function='Poisson', iterations=2500,
        learning_rate  =p.get('learning_rate',   0.03),
        depth          =min(p.get('max_depth',    6),8),
        l2_leaf_reg    =p.get('reg_lambda',       3.0),
        subsample      =p.get('subsample',        0.85),
        bootstrap_type='Bernoulli',
        random_seed=seed, verbose=0,
        allow_writing_files=False, early_stopping_rounds=150)

print('Booster param factories ready ✓')

In [ ]:
oof_team  = {k: np.zeros(n_train) for k in ['lgb','xgb','cat']}
oof_opp   = {k: np.zeros(n_train) for k in ['lgb','xgb','cat']}
test_team = {k: np.zeros(n_test)  for k in ['lgb','xgb','cat']}
test_opp  = {k: np.zeros(n_test)  for k in ['lgb','xgb','cat']}
oof_mask  = np.zeros(n_train, dtype=bool)

def _fit_lgb(X_tr,y_tr,w_tr,X_val,y_val,seed,gender,target):
    m = lgb.LGBMRegressor(**make_lgb_params(seed,gender,target))
    m.fit(X_tr,y_tr,sample_weight=w_tr,eval_set=[(X_val,y_val)],
          callbacks=[lgb.early_stopping(150,verbose=False)])
    return m

def _fit_xgb(X_tr,y_tr,w_tr,X_val,y_val,seed,gender,target):
    m = xgb.XGBRegressor(**make_xgb_params(seed,gender,target))
    m.fit(X_tr,y_tr,sample_weight=w_tr,eval_set=[(X_val,y_val)],verbose=False)
    return m

def _fit_cat(X_tr,y_tr,w_tr,X_val,y_val,seed,gender,target):
    m = CatBoostRegressor(**make_cat_params(seed,gender,target))
    m.fit(X_tr,y_tr,sample_weight=w_tr,eval_set=(X_val,y_val),
          use_best_model=True,verbose=False)
    return m

FITTERS = {'lgb':_fit_lgb, 'xgb':_fit_xgb, 'cat':_fit_cat}

for gender in ['M','W']:
    g_tr_mask = gender_train == gender
    g_te_mask = gender_test  == gender
    if g_tr_mask.sum() < 500: continue

    print(f'\n{"═"*60}')
    print(f'  Gender = {gender}  (train={g_tr_mask.sum()}, test={g_te_mask.sum()})')
    print(f'{"═"*60}')

    g_train_fe  = train_fe[g_tr_mask].reset_index(drop=True)
    X_g         = X[g_tr_mask]
    yt_g        = y_team[g_tr_mask]; yo_g = y_opp[g_tr_mask]
    w_g         = fit_weights_train[g_tr_mask]
    X_test_g    = X_test[g_te_mask]
    orig_tr_idx = np.where(g_tr_mask)[0]
    orig_te_idx = np.where(g_te_mask)[0]
    folds_g     = build_temporal_match_folds(g_train_fe, n_splits=5, holdout_start_frac=0.55)

    for fold_i,(tr_idx,val_idx) in enumerate(folds_g, start=1):
        X_tr,X_val = X_g[tr_idx],X_g[val_idx]
        yt_tr=yt_g[tr_idx]; yo_tr=yo_g[tr_idx]
        yt_val=yt_g[val_idx]; yo_val=yo_g[val_idx]
        w_tr=w_g[tr_idx]

        for booster,fitter in FITTERS.items():
            m_t = fitter(X_tr,yt_tr,w_tr,X_val,yt_val,SEED,gender,'team')
            m_o = fitter(X_tr,yo_tr,w_tr,X_val,yo_val,SEED,gender,'opp')
            oof_team[booster][orig_tr_idx[val_idx]] = m_t.predict(X_val)
            oof_opp[booster] [orig_tr_idx[val_idx]] = m_o.predict(X_val)
            test_team[booster][orig_te_idx] += m_t.predict(X_test_g) / len(folds_g)
            test_opp[booster] [orig_te_idx] += m_o.predict(X_test_g) / len(folds_g)
            joblib.dump(m_t, f'{MODEL_DIR}/{gender}_fold{fold_i}_{booster}_team.pkl')
            joblib.dump(m_o, f'{MODEL_DIR}/{gender}_fold{fold_i}_{booster}_opp.pkl')

        oof_mask[orig_tr_idx[val_idx]] = True
        vd = g_train_fe.iloc[val_idx]['date']
        print(f'  [{gender}] fold {fold_i} | train={len(tr_idx):5d} | val={len(val_idx):5d} '
              f'| {vd.min().date()} → {vd.max().date()}')

print('\n✓ All base models trained.')
print(f'OOF coverage: {oof_mask.sum()}/{n_train} ({100*oof_mask.mean():.1f}%)')

## 9. Ridge Stacking Meta-Learner

In [ ]:
stacker_team = {}; stacker_opp = {}; stacker_scaler = {}

for gender in ['M','W']:
    g_mask = oof_mask & (gender_train == gender)
    if g_mask.sum() < 100: continue

    Z_team_oof = np.column_stack([oof_team['lgb'][g_mask], oof_team['xgb'][g_mask], oof_team['cat'][g_mask]])
    Z_opp_oof  = np.column_stack([oof_opp['lgb'][g_mask],  oof_opp['xgb'][g_mask],  oof_opp['cat'][g_mask]])
    yt_g = y_team[g_mask]; yo_g = y_opp[g_mask]
    w_g  = official_weights_train[g_mask]

    scaler = StandardScaler()
    Z_team_scaled = scaler.fit_transform(Z_team_oof)
    Z_opp_scaled  = scaler.transform(Z_opp_oof)

    ridge_t = Ridge(alpha=1.0, positive=True, fit_intercept=True)
    ridge_o = Ridge(alpha=1.0, positive=True, fit_intercept=True)
    ridge_t.fit(Z_team_scaled, yt_g, sample_weight=w_g)
    ridge_o.fit(Z_opp_scaled,  yo_g, sample_weight=w_g)

    stacker_team[gender]   = ridge_t
    stacker_opp[gender]    = ridge_o
    stacker_scaler[gender] = scaler

    ct_n = ridge_t.coef_ / (ridge_t.coef_.sum() or 1)
    co_n = ridge_o.coef_ / (ridge_o.coef_.sum() or 1)
    print(f'[{gender}] team coefs: LGB={ct_n[0]:.3f}  XGB={ct_n[1]:.3f}  CAT={ct_n[2]:.3f}')
    print(f'[{gender}] opp  coefs: LGB={co_n[0]:.3f}  XGB={co_n[1]:.3f}  CAT={co_n[2]:.3f}')

print('Ridge stacking ✓')

In [ ]:
oof_team_blend  = np.zeros(n_train)
oof_opp_blend   = np.zeros(n_train)
test_team_blend = np.zeros(n_test)
test_opp_blend  = np.zeros(n_test)

for gender in ['M','W']:
    mask_tr = oof_mask & (gender_train == gender)
    mask_te = gender_test == gender
    if gender not in stacker_team:
        mask_tr2 = gender_train == gender
        for bst in ['lgb','xgb','cat']:
            oof_team_blend[mask_tr2] += oof_team[bst][mask_tr2] / 3
            oof_opp_blend[mask_tr2]  += oof_opp[bst][mask_tr2]  / 3
            test_team_blend[mask_te] += test_team[bst][mask_te] / 3
            test_opp_blend[mask_te]  += test_opp[bst][mask_te]  / 3
        continue
    scaler = stacker_scaler[gender]
    Z_t_oof = np.column_stack([oof_team['lgb'][mask_tr], oof_team['xgb'][mask_tr], oof_team['cat'][mask_tr]])
    Z_o_oof = np.column_stack([oof_opp['lgb'][mask_tr],  oof_opp['xgb'][mask_tr],  oof_opp['cat'][mask_tr]])
    oof_team_blend[mask_tr] = stacker_team[gender].predict(scaler.transform(Z_t_oof))
    oof_opp_blend[mask_tr]  = stacker_opp[gender].predict(scaler.transform(Z_o_oof))
    Z_t_te = np.column_stack([test_team['lgb'][mask_te], test_team['xgb'][mask_te], test_team['cat'][mask_te]])
    Z_o_te = np.column_stack([test_opp['lgb'][mask_te],  test_opp['xgb'][mask_te],  test_opp['cat'][mask_te]])
    test_team_blend[mask_te] = stacker_team[gender].predict(scaler.transform(Z_t_te))
    test_opp_blend[mask_te]  = stacker_opp[gender].predict(scaler.transform(Z_o_te))

oof_team_blend, oof_opp_blend = enforce_match_consistency_vec(train_fe, oof_team_blend, oof_opp_blend)
test_team_blend, test_opp_blend = enforce_match_consistency_vec(test_fe, test_team_blend, test_opp_blend)

oof_team_blend  = np.clip(oof_team_blend,  1e-4, MAX_GOALS)
oof_opp_blend   = np.clip(oof_opp_blend,   1e-4, MAX_GOALS)
test_team_blend = np.clip(test_team_blend, 1e-4, MAX_GOALS)
test_opp_blend  = np.clip(test_opp_blend,  1e-4, MAX_GOALS)

print(f'Blended λ — train: team={oof_team_blend[oof_mask].mean():.3f}, opp={oof_opp_blend[oof_mask].mean():.3f}')
print(f'Blended λ — test:  team={test_team_blend.mean():.3f}, opp={test_opp_blend.mean():.3f}')

## 10. ZIP-aware MBR Decoding + Per-Gender Dixon-Coles ρ

**V6 upgrade:** MBR decoding menggunakan ZIP PMF (bukan pure Poisson), menggabungkan:
1. P(goals=0) dari hurdle classifier
2. Poisson λ dari ensemble regressor
3. Extended 6-cell Dixon-Coles correction

In [ ]:
def build_loss_tensor(max_goals=MAX_GOALS):
    S = max_goals + 1
    L = np.zeros((S,S,S,S), dtype=np.float64)
    for t in range(S):
        for o in range(S):
            for a in range(S):
                for b in range(S):
                    L[t,o,a,b] = official_match_loss([int(t)],[int(o)],[int(a)],[int(b)])[0]
    return L

LOSS_TENSOR = build_loss_tensor()
S = MAX_GOALS + 1
SUPPORT = np.arange(S)
LOG_PMF_PREFIX = np.array([np.sum(np.log(np.arange(1, k+1))) for k in SUPPORT])

def _poisson_pmf_matrix(lam, support=SUPPORT):
    lam = np.asarray(lam, dtype=np.float64)[:, None]
    k   = support[None, :]
    log_p = k * np.log(lam) - lam - LOG_PMF_PREFIX[None, :]
    p = np.exp(log_p)
    p /= p.sum(axis=1, keepdims=True)
    return p

# ─── V6 NEW: ZIP PMF ───────────────────────────────────────────────────
def _zip_pmf_matrix(lam, p_zero, support=SUPPORT):
    """
    Zero-Inflated Poisson PMF:
    P(Y=0) = p_zero + (1 - p_zero) * exp(-lambda)
    P(Y=k) = (1 - p_zero) * Poisson(k; lambda),  k > 0
    """
    lam    = np.asarray(lam,    dtype=np.float64)
    p_zero = np.clip(np.asarray(p_zero, dtype=np.float64), 0.0, 0.99)
    p_pois = _poisson_pmf_matrix(lam)           # (n, S)
    p_zip  = (1.0 - p_zero)[:, None] * p_pois  # scale all bins
    p_zip[:, 0] += p_zero                       # add structural zeros
    p_zip = np.clip(p_zip, 1e-12, None)
    p_zip /= p_zip.sum(axis=1, keepdims=True)
    return p_zip

def mbr_decode_zip(lam_team, lam_opp, p0_team, p0_opp, rho=0.0):
    """V6: ZIP-aware MBR dengan extended 6-cell Dixon-Coles correction."""
    p_t = _zip_pmf_matrix(lam_team, p0_team)
    p_o = _zip_pmf_matrix(lam_opp,  p0_opp)
    P   = p_t[:, :, None] * p_o[:, None, :]  # (n, S, S)

    if rho != 0.0:
        lt = np.asarray(lam_team, dtype=np.float64)
        lo = np.asarray(lam_opp,  dtype=np.float64)
        P[:, 0, 0] *= (1 - lt * lo * rho)
        P[:, 0, 1] *= (1 + lt * rho)
        P[:, 1, 0] *= (1 + lo * rho)
        P[:, 1, 1] *= (1 - rho)
        rho2 = rho * 0.5
        P[:, 2, 0] *= np.clip(1 + lo * rho2, 1e-6, None)
        P[:, 0, 2] *= np.clip(1 + lt * rho2, 1e-6, None)
        P = np.clip(P, 1e-12, None)
        P = P / P.sum(axis=(1,2), keepdims=True)

    E_L  = np.einsum('ito,toab->iab', P, LOSS_TENSOR)
    flat = E_L.reshape(P.shape[0], -1).argmin(axis=1)
    return (flat // S).astype(int), (flat % S).astype(int)

print('Loss tensor built ✓ | ZIP MBR decoder defined ✓')

In [ ]:
# ─── Per-gender ρ via scipy (on OOF, no leak) ─────────────────────────
covered_mask = oof_mask
y_t_oof  = y_team[covered_mask]; y_o_oof  = y_opp[covered_mask]
w_oof    = official_weights_train[covered_mask]
lt_oof   = oof_team_blend[covered_mask]; lo_oof = oof_opp_blend[covered_mask]
p0t_oof  = oof_p0_team[covered_mask];   p0o_oof = oof_p0_opp[covered_mask]
g_oof    = gender_train[covered_mask]

best_rho_per_gender = {}

for gender in ['M','W']:
    g_mask = g_oof == gender
    if g_mask.sum() < 50: best_rho_per_gender[gender] = 0.0; continue

    yt_g=y_t_oof[g_mask]; yo_g=y_o_oof[g_mask]; wg=w_oof[g_mask]
    lt_g=lt_oof[g_mask];  lo_g=lo_oof[g_mask]
    p0t_g=p0t_oof[g_mask]; p0o_g=p0o_oof[g_mask]

    def neg_score(rho):
        a,b = mbr_decode_zip(lt_g, lo_g, p0t_g, p0o_g, rho=float(rho))
        return awmae_score(yt_g, yo_g, a, b, weights=wg)

    result = minimize_scalar(neg_score, bounds=(-0.25, 0.25), method='bounded',
                             options={'xatol':1e-4,'maxiter':100})
    best_rho_per_gender[gender] = float(result.x)
    score_zero = awmae_score(yt_g,yo_g,*mbr_decode_zip(lt_g,lo_g,p0t_g,p0o_g,rho=0.0),weights=wg)
    print(f'[{gender}]  ρ = {result.x:+.4f}  '
          f'AW-MAE = {result.fun:.5f}  '
          f'(ρ=0: {score_zero:.5f}, Δ={score_zero-result.fun:+.5f})')

print(f'\nPer-gender ρ: {best_rho_per_gender}')

In [ ]:
# ─── Apply ZIP MBR decoding to test ───────────────────────────────────
pred_team_int = np.zeros(n_test, dtype=int)
pred_opp_int  = np.zeros(n_test, dtype=int)

for gender in ['M','W']:
    g_mask = gender_test == gender
    rho_g  = best_rho_per_gender.get(gender, 0.0)
    a_g, b_g = mbr_decode_zip(
        test_team_blend[g_mask], test_opp_blend[g_mask],
        test_p0_team[g_mask],    test_p0_opp[g_mask],
        rho=rho_g)
    pred_team_int[g_mask] = a_g
    pred_opp_int[g_mask]  = b_g

print(f'V6 predictions ready.')
print(f'Prediction distribution (team_pred):')
print(pd.Series(pred_team_int).value_counts().sort_index().to_string())
print(f'\nPrediction distribution (opp_pred):')
print(pd.Series(pred_opp_int).value_counts().sort_index().to_string())

## 11. Evaluasi — V5 vs V6 vs Baseline vs Oracle

In [ ]:
gt_map = gt[['Id','team_goals','opp_goals']].copy()
test_with_gt = test.merge(gt_map, on='Id', how='left', validate='one_to_one')
tw = test_with_gt['tournament'].apply(get_tournament_weight).values

gm_int = int(round(train['team_goals'].mean()))
baseline_aw = awmae_score(test_with_gt['team_goals'].values, test_with_gt['opp_goals'].values,
                           np.full(len(test_with_gt),gm_int), np.full(len(test_with_gt),gm_int), weights=tw)

rc_team = round_clip(test_team_blend); rc_opp = round_clip(test_opp_blend)
roundclip_aw = awmae_score(test_with_gt['team_goals'].values, test_with_gt['opp_goals'].values,
                            rc_team, rc_opp, weights=tw)

model_aw = awmae_score(test_with_gt['team_goals'].values, test_with_gt['opp_goals'].values,
                        pred_team_int, pred_opp_int, weights=tw)

oracle_aw = awmae_score(test_with_gt['team_goals'].values, test_with_gt['opp_goals'].values,
                         test_with_gt['team_goals'].values, test_with_gt['opp_goals'].values, weights=tw)

print(f'{"─"*72}')
print(f'  Baseline AW-MAE (predict {gm_int}-{gm_int})    : {baseline_aw:.4f}')
print(f'  Round-clip ref                       : {roundclip_aw:.4f}')
print(f'  V6 full (ZIP+Stack+MBR+ρ+No-leak)   : {model_aw:.4f}  ← submitted')
print(f'  Oracle (ground truth floor)          : {oracle_aw:.6f}')
print(f'{"─"*72}')
print(f'  Improvement vs baseline : {100*(baseline_aw-model_aw)/baseline_aw:+.2f}%')
print(f'  Improvement MBR vs clip : {100*(roundclip_aw-model_aw)/roundclip_aw:+.2f}%')
print(f'  Target (< 2.0): {"✅ ACHIEVED" if model_aw < 2.0 else f"❌ {model_aw:.4f} — lebih dekat!"}')

In [ ]:
TW_LABELS = {2.00:'World Cup (2.0)', 1.80:'Asian Cup/AFC (1.8)',
             1.20:'Default (1.2)',   0.96:'Friendly (0.96)'}

breakdown = test_with_gt.copy()
breakdown['t_weight']      = tw
breakdown['wlabel']        = breakdown['t_weight'].map(TW_LABELS).fillna('Other')
breakdown['match_loss_v6'] = official_match_loss(breakdown['team_goals'], breakdown['opp_goals'],
                                                  pred_team_int, pred_opp_int)
breakdown['match_loss_rc'] = official_match_loss(breakdown['team_goals'], breakdown['opp_goals'],
                                                  rc_team, rc_opp)

print('Loss breakdown (gender × tournament):')
agg = breakdown.groupby(['gender','wlabel'])['match_loss_v6'].mean()
print(agg.sort_values(ascending=False).to_string())

fig, ax = plt.subplots(figsize=(12, 5))
agg2 = breakdown.groupby(['gender','wlabel'])['match_loss_v6'].mean().reset_index()
colors = {'M':'#2980b9','W':'#e91e8c'}
wlabels = [v for v in TW_LABELS.values() if v in agg2['wlabel'].values]
x = np.arange(len(wlabels)); width = 0.35
for i,g in enumerate(['M','W']):
    vals = [agg2[(agg2['gender']==g)&(agg2['wlabel']==w)]['match_loss_v6'].values for w in wlabels]
    vals = [v[0] if len(v) else 0.0 for v in vals]
    bars = ax.bar(x+(i-0.5)*width, vals, width, label=g, color=colors[g], edgecolor='white')
    for bar,v in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                f'{v:.3f}', ha='center', fontsize=8, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(wlabels, rotation=10)
ax.set_ylabel('Avg match loss (V6)'); ax.legend()
ax.set_title(f'V6 Loss Breakdown by Gender × Tournament  (AW-MAE={model_aw:.4f})', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
def decompose_loss(y_t,y_o,p_t,p_o):
    y_t=np.asarray(y_t).astype(int); y_o=np.asarray(y_o).astype(int)
    p_t=np.asarray(p_t).astype(int); p_o=np.asarray(p_o).astype(int)
    base_mae  = (np.abs(y_t-p_t)+np.abs(y_o-p_o))/2.0
    exact     = (y_t==p_t)&(y_o==p_o)
    outcome_h = (np.sign(y_t-y_o)==np.sign(p_t-p_o))
    gd_hit    = ((y_t-y_o)==(p_t-p_o))
    return pd.DataFrame({'base_mae':base_mae,'exact_hit':exact.astype(int),
                         'outcome_hit':outcome_h.astype(int),'gd_hit':gd_hit.astype(int)})

decomp = decompose_loss(test_with_gt['team_goals'],test_with_gt['opp_goals'],
                        pred_team_int, pred_opp_int)
decomp['gender'] = test_with_gt['gender'].values
print('V6 hit rates by gender:')
for g in ['M','W']:
    sub = decomp[decomp['gender']==g]
    print(f'  [{g}]  n={len(sub):5d}  '
          f'exact={sub["exact_hit"].mean():.3f}  '
          f'outcome={sub["outcome_hit"].mean():.3f}  '
          f'GD={sub["gd_hit"].mean():.3f}  '
          f'base_MAE={sub["base_mae"].mean():.3f}')
print(f'\n  [ALL] exact={decomp["exact_hit"].mean():.3f}  '
      f'outcome={decomp["outcome_hit"].mean():.3f}  '
      f'GD={decomp["gd_hit"].mean():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, gender, color in zip(axes, ['M','W'], ['#2980b9','#e91e8c']):
    try:
        m = joblib.load(f'{MODEL_DIR}/{gender}_fold5_lgb_team.pkl')
        fi = (pd.DataFrame({'feature':feature_cols,'importance':m.feature_importances_})
                .sort_values('importance', ascending=False).head(25))
        ax.barh(fi['feature'][::-1], fi['importance'][::-1], color=color)
        ax.set_title(f'Top-25 Features — {gender} (LGB team_goals)', fontweight='bold')
        ax.set_xlabel('LightGBM importance')
        for label in ax.get_yticklabels():
            txt = label.get_text()
            if any(x in txt for x in ['home_avg','away_avg','home_win','away_win',
                                       'gdp_miss','pop_miss','h2h_gd_last10','h2h_win_rate']):
                label.set_color('darkgreen'); label.set_fontweight('bold')
    except Exception as e:
        ax.text(0.5,0.5,str(e),ha='center',transform=ax.transAxes)
plt.suptitle('Green = NEW V6 features', fontsize=11)
plt.tight_layout(); plt.show()

## 12. Generate Submission

In [ ]:
submission = test_fe[['Id']].copy()
submission['team_goals'] = pred_team_int
submission['opp_goals']  = pred_opp_int

assert submission.isna().sum().sum() == 0, 'NaNs in submission!'
assert (submission['team_goals'] >= 0).all(), 'Negative goals!'
assert (submission['opp_goals']  >= 0).all(), 'Negative goals!'
assert len(submission) == len(test), 'Row count mismatch!'

submission.to_csv('submission_v6.csv', index=False)

sub_check = submission.merge(gt_map, on='Id', suffixes=('_pred','_true'))
tw_final  = test.set_index('Id').loc[sub_check['Id'],'tournament'] \
                .apply(get_tournament_weight).values
final_score = awmae_score(sub_check['team_goals_true'], sub_check['opp_goals_true'],
                           sub_check['team_goals_pred'], sub_check['opp_goals_pred'],
                           weights=tw_final)

fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax,col in zip(axes,['team_goals','opp_goals']):
    counts = submission[col].value_counts().sort_index().head(12)
    ax.bar(counts.index, counts.values, color='#27ae60', edgecolor='white')
    ax.set_title(f'{col} Distribution (V6)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Goals'); ax.set_ylabel('Count')
    for i,v in zip(counts.index,counts.values):
        ax.text(i,v+20,str(v),ha='center',fontsize=8)
plt.suptitle(f'V6 Submission — AW-MAE: {final_score:.5f}', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print('='*72)
print('  V6 PIPELINE COMPLETE')
print('='*72)
print(f'  Baseline AW-MAE ({gm_int}-{gm_int})              : {baseline_aw:.4f}')
print(f'  V6 round-clip ref                      : {roundclip_aw:.4f}')
print(f'  V6 full                                : {model_aw:.4f}  ← submitted')
print(f'  Oracle (metric floor)                  : {oracle_aw:.6f}')
print(f'{"─"*72}')
print(f'  Per-gender Dixon-Coles ρ: M={best_rho_per_gender.get("M",0):.4f}  W={best_rho_per_gender.get("W",0):.4f}')
print(f'  ZIP hurdle P0 (test avg): team={test_p0_team.mean():.3f}, opp={test_p0_opp.mean():.3f}')
print(f'  Models saved in: {MODEL_DIR}/')
print('='*72)